# Notebook 08 — Module Assessment

**Module:** 11 — Structural Biology  
**Tier:** 3 — Survey  
**Notebook:** 08 of 08  
**Time estimate:** 2–3 hours

---
## Instructions

**Total: 50 points**

This is a Tier 3 module. The assessment tests:
1. Conceptual understanding (20 pts) — you can explain the core ideas
2. Practical coding (30 pts) — you can implement the key algorithms

Close NB01–NB07 before starting. Implement from memory.
If you cannot complete a section, write what you understand and what is fuzzy.

---
## Section A — Conceptual Questions (20 points)

**A1 (4 pts):** Describe the four levels of protein structure. For each level, name
one stabilizing interaction.

**A2 (4 pts):** Compare X-ray crystallography and cryo-EM. What is the main advantage
of cryo-EM for studying membrane proteins? What caused the cryo-EM resolution
revolution after 2013?

**A3 (4 pts):** Explain the pLDDT score from AlphaFold2:
- What does it measure?
- What does pLDDT = 35 indicate?
- What does pLDDT = 95 indicate?
- What is the relationship between low pLDDT regions and protein disorder?

**A4 (4 pts):** A protein has Ramachandran statistics:
- 85% residues in "most favored" region
- 12% in "additionally allowed"
- 3% in "generously allowed"
- 0% in disallowed

Is this a high-quality or low-quality structure? Explain.

**A5 (4 pts):** Explain the Kabsch algorithm in plain language. Why is superposition
necessary before computing RMSD? What mathematical operation is at its core?

---
**[Write your answers to A1–A5 here as markdown text below each question.]**

**A1 answer:** *[Write here.]*

**A2 answer:** *[Write here.]*

**A3 answer:** *[Write here.]*

**A4 answer:** *[Write here.]*

**A5 answer:** *[Write here.]*

---
## Section B — Coding Tasks (30 points)

### B1 — PDB Parser (8 points)

Implement `parse_pdb_line(line)` → `Atom` dataclass.
Implement `get_ca_coords(atoms)` → `(residue_numbers, coords)` where `coords` is (N, 3).
Demonstrate: parse a synthetic PDB string and extract all Cα coordinates.

In [ ]:
# B1 — PDB Parser
import numpy as np
from dataclasses import dataclass
from typing import Optional, List

# Test PDB string (paste from NB03 or write from memory)
TEST_PDB = """ATOM      1  N   ALA A   1      -0.525   1.362   0.000  1.00 12.31           N
ATOM      2  CA  ALA A   1       0.000   0.000   0.000  1.00  8.50           C
ATOM      3  C   ALA A   1       1.522   0.000   0.000  1.00  9.10           C
ATOM      4  O   ALA A   1       2.175   1.062   0.000  1.00  7.80           O
ATOM      5  N   GLY A   2       2.142  -1.162   0.230  1.00 11.20           N
ATOM      6  CA  GLY A   2       3.590  -1.305   0.360  1.00 10.15           C
ATOM      7  C   GLY A   2       4.100  -0.960   1.750  1.00  9.80           C
ATOM      8  O   GLY A   2       3.350  -0.480   2.600  1.00  8.90           O
"""

raise NotImplementedError('Implement parse_pdb_line() and get_ca_coords() from scratch here')

### B2 — Dihedral Angle (8 points)

Implement `dihedral_angle(p1, p2, p3, p4)` → angle in degrees.
Verify: for an ideal helix, φ should be approximately −60° and ψ approximately −40°.

In [ ]:
# B2 — Dihedral angle
raise NotImplementedError('Implement dihedral_angle() here and verify with helix test')

### B3 — Contact Map (6 points)

Implement `contact_map(coords, threshold=8.0, min_seqsep=5)` → binary (N, N) matrix.
Apply it to a 60-residue synthetic Cβ trace (helix). Visualize and report the
number of contacts.

In [ ]:
# B3 — Contact map
import matplotlib.pyplot as plt
raise NotImplementedError('Implement contact_map() and visualize here')

### B4 — Kabsch RMSD (8 points)

Implement `kabsch_rmsd(A, B)` → scalar RMSD after optimal superposition.
Verify: RMSD = 0 for identical structures in different orientations.

In [ ]:
# B4 — Kabsch RMSD
raise NotImplementedError('Implement kabsch_rmsd() here')

---
## Reference Implementations (hidden — reveal only after attempting)

<!--
B1:
  @dataclass
  class Atom:
      serial: int; name: str; altloc: str; res_name: str; chain: str
      res_seq: int; icode: str; x: float; y: float; z: float
      occupancy: float; b_factor: float; element: str; record: str
      @property
      def coord(self): return np.array([self.x, self.y, self.z])

  def parse_pdb_line(line):
      record = line[0:6].strip()
      if record not in ('ATOM', 'HETATM'): return None
      serial   = int(line[6:11])
      name     = line[12:16].strip()
      altloc   = line[16].strip()
      res_name = line[17:20].strip()
      chain    = line[21].strip()
      res_seq  = int(line[22:26])
      icode    = line[26].strip()
      x, y, z  = float(line[30:38]), float(line[38:46]), float(line[46:54])
      occ = float(line[54:60]) if line[54:60].strip() else 1.0
      bfac = float(line[60:66]) if line[60:66].strip() else 0.0
      elem = line[76:78].strip() if len(line) > 76 else ''
      return Atom(serial,name,altloc,res_name,chain,res_seq,icode,x,y,z,occ,bfac,elem,record)

B2:
  def dihedral_angle(p1, p2, p3, p4):
      b1 = np.array(p2)-np.array(p1)
      b2 = np.array(p3)-np.array(p2)
      b3 = np.array(p4)-np.array(p3)
      n1 = np.cross(b1, b2); n2 = np.cross(b2, b3)
      if np.linalg.norm(n1)<1e-9 or np.linalg.norm(n2)<1e-9: return np.nan
      n1/=np.linalg.norm(n1); n2/=np.linalg.norm(n2)
      b2h = b2/np.linalg.norm(b2)
      m1 = np.cross(n1, b2h)
      return np.degrees(np.arctan2(np.dot(m1,n2), np.dot(n1,n2)))

B3:
  def contact_map(coords, threshold=8.0, min_seqsep=5):
      diff = coords[:,None,:] - coords[None,:,:]
      D = np.sqrt((diff**2).sum(axis=-1))
      C = (D <= threshold).astype(int)
      for k in range(-min_seqsep+1, min_seqsep):
          np.fill_diagonal(C[max(0,-k):, max(0,k):], 0)
      return C

B4:
  def kabsch_rmsd(A, B):
      A, B = np.array(A,dtype=float), np.array(B,dtype=float)
      cA, cB = A.mean(0), B.mean(0)
      P, Q = A-cA, B-cB
      H = P.T @ Q
      U, S, Vt = np.linalg.svd(H)
      d = np.linalg.det(Vt.T @ U.T)
      R = Vt.T @ np.diag([1,1,d]) @ U.T
      return np.sqrt(((P@R.T - Q)**2).sum(1).mean()), R, cA, cB
-->

---

## Module 11 Completion Checklist

- [ ] NB01: Can explain four levels of protein structure and backbone dihedral angles
- [ ] NB02: Can compare X-ray, cryo-EM, and NMR; knows B-factor meaning
- [ ] NB03: Can parse PDB ATOM records and extract Cα coordinates
- [ ] NB04: Can compute dihedral angles and produce a Ramachandran plot
- [ ] NB05: Can compute a contact map and explain its connection to AlphaFold2
- [ ] NB06: Can explain AlphaFold2 at interview level (Evoformer, pLDDT, limitations)
- [ ] NB07: Can implement Kabsch RMSD; understands structure-based drug design pipeline
- [ ] NB08: Assessment complete
- [ ] Paper: Jumper et al. 2021 (AlphaFold2) — Pass 1 + Pass 2 complete; log in paper-notes/
- [ ] Track A: Can answer "how does AlphaFold2 work" in an interview without notes